In [0]:
from pyspark.sql import functions as F

CATALOG = "dbw_taxrisk_dev"

p  = spark.table(f"{CATALOG}.silver.posts_classified")
cr = spark.table(f"{CATALOG}.silver.creators")

def cnt(cls):
    return F.sum((F.col("pred_class") == cls).cast("int"))

def val(cls):
    return F.coalesce(F.sum(F.when(F.col("pred_class") == cls, F.col("value_mid"))), F.lit(0.0))

by_class = p.groupBy("creator_key").agg(
    F.count("*").alias("n_posts"),
    cnt("paid").alias("n_paid"),
    cnt("disclosed_collab").alias("n_collab"),
    cnt("in_kind").alias("n_in_kind"),
    cnt("unclear").alias("n_unclear"),
    val("in_kind").alias("val_in_kind_mid"),
    val("disclosed_collab").alias("val_collab_mid"),
    val("unclear").alias("val_unclear_mid"),
    F.min("posted_at").alias("first_post_at"),
    F.max("posted_at").alias("last_post_at"),
)

# Recurrence: how often the same creator returns to the same venue
per_venue = (p.filter("venue_id IS NOT NULL")
             .groupBy("creator_key", "venue_id").agg(F.count("*").alias("n")))
recurrence = per_venue.groupBy("creator_key").agg(
    F.countDistinct("venue_id").alias("distinct_venues"),
    F.max("n").alias("max_posts_same_venue"),
    F.sum((F.col("n") >= 2).cast("int")).alias("venues_repeated"),
)

features = (by_class
    .join(recurrence, "creator_key", "left")
    .join(cr, "creator_key", "left")
    .fillna(0, subset=["distinct_venues", "max_posts_same_venue", "venues_repeated"])
    .withColumn("venue_post_share",
                (F.col("n_in_kind") + F.col("n_collab") + F.col("n_unclear")) / F.col("n_posts")))

(features.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.creator_features"))

Validation

In [0]:
g = spark.table(f"{CATALOG}.gold.creator_features")
print("creadores:", g.count(), "| suma de n_posts:", g.agg(F.sum("n_posts")).first()[0])
display(g.orderBy(F.desc("val_in_kind_mid")).select(
    "creator_key", "tier", "n_posts", "n_in_kind", "n_collab", "n_unclear",
    "distinct_venues", "max_posts_same_venue", "val_in_kind_mid", "declared_income").limit(10))
